# Shared-Split Evaluation of Polymer Property Models and Molecular Representations

## Goal

Compare individual regression models for five polymer properties using
one shared row-level training/validation split.

Evaluate molecular representations with 9, 13, and 21 descriptors,
and 1,024-position Morgan fingerprints. Perform targeted nested
hyperparameter tuning for radius of gyration (Rg).

## Reproducibility

Place the original training dataset at `data/raw/train.csv`.
The notebook calculates the required descriptors directly from SMILES.

Use the Python environment belonging to this repository.
Results are written to `results/06_shared_split/`.

## Evaluation scope

All model selection uses cross-validation within the training partition.
The shared outer validation partition remains unused in this notebook.

These are development results. Previous analyses have examined this
dataset, so this partition is not an independent final test.

Predicting missing target labels and training a joint model for all
five properties remain future steps.

In [8]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

# Locate the project root.
def find_project_root(start):
    for folder in [start, *start.parents]:
        if (
            (folder / "pyproject.toml").is_file()
            and (folder / "notebooks").is_dir()
        ):
            return folder
    raise FileNotFoundError("Project root not found.")


PROJECT_ROOT = find_project_root(Path.cwd().resolve())

# Verify that the notebook uses this project's environment.
assert Path(sys.prefix).resolve() == (
    PROJECT_ROOT / ".venv"
).resolve(), "Select the Python kernel from this project's .venv."

TARGETS = ["Tg", "FFV", "Tc", "Density", "Rg"]
RANDOM_STATE = 42
VALIDATION_FRACTION = 0.20

# Build the required feature table from the shared project's raw data.
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.descriptors import calculate_descriptor_table

raw_path = PROJECT_ROOT / "data" / "raw" / "train.csv"

if not raw_path.is_file():
    raise FileNotFoundError(
        "Place the original training dataset at data/raw/train.csv."
    )

raw_train = pd.read_csv(raw_path)
raw_train.index.name = "row_index"

required_columns = ["id", "SMILES", *TARGETS]
missing_columns = set(required_columns) - set(raw_train.columns)

if missing_columns:
    raise ValueError(f"Missing input columns: {sorted(missing_columns)}")

# Preserve all original rows, their order, and their target labels.
base_descriptors = calculate_descriptor_table(
    raw_train["SMILES"],
    extended=True,
)

assert base_descriptors.index.equals(raw_train.index)
assert np.isfinite(base_descriptors.to_numpy(dtype=float)).all()

table = pd.concat(
    [
        raw_train[required_columns].copy(),
        base_descriptors,
    ],
    axis=1,
)

table.index.name = "row_index"

pd.testing.assert_frame_equal(
    table[TARGETS],
    raw_train[TARGETS],
)

print("Input data:", raw_path)
print("Calculated descriptors:", base_descriptors.shape)

assert table.index.is_unique
assert table["id"].is_unique
assert table[TARGETS].notna().any(axis=1).all()

print("Project:", PROJECT_ROOT)
print("Python:", sys.executable)
print("Feature table:", table.shape)

Input data: C:\Users\EMS-User\capstone_virtual_lab_mohamed_suhmi\data\raw\train.csv
Calculated descriptors: (7973, 13)
Project: C:\Users\EMS-User\capstone_virtual_lab_mohamed_suhmi
Python: c:\Users\EMS-User\capstone_virtual_lab_mohamed_suhmi\.venv\Scripts\python.exe
Feature table: (7973, 20)


In [9]:
# Split all structure indices once, before selecting labeled rows.
training_indices, validation_indices = train_test_split(
    table.index.to_numpy(),
    test_size=VALIDATION_FRACTION,
    random_state=RANDOM_STATE,
    shuffle=True,
)

training_indices = pd.Index(training_indices)
validation_indices = pd.Index(validation_indices)

# Verify that the partitions are disjoint and complete.
assert training_indices.intersection(validation_indices).empty
assert len(training_indices) + len(validation_indices) == len(table)

# Preserve the original labels, including missing values.
y_training_original = table.loc[training_indices, TARGETS].copy()
y_validation_original = table.loc[validation_indices, TARGETS].copy()

# Inspect the available reference labels in each partition.
shared_split_summary = pd.DataFrame({
    "training_observed": y_training_original.notna().sum(),
    "training_missing": y_training_original.isna().sum(),
    "validation_observed": y_validation_original.notna().sum(),
    "validation_missing": y_validation_original.isna().sum(),
})

assert (shared_split_summary["training_observed"] >= 5).all()
assert (shared_split_summary["validation_observed"] >= 2).all()

print("Training structures:", len(training_indices))
print("Validation structures:", len(validation_indices))
display(shared_split_summary)

Training structures: 6378
Validation structures: 1595


,training_observed,training_missing,validation_observed,validation_missing
Tg,424,5954,87,1508
FFV,5611,767,1419,176
Tc,592,5786,145,1450
Density,490,5888,123,1472
Rg,490,5888,124,1471


## 2. Save the shared row-level split

The dataset contains 6,378 training structures and 1,595 validation
structures. Each structure has the same assignment across all five targets.

Target labels remain unchanged, including missing values.
Models are evaluated only against originally available validation labels.

The split is saved separately from previous target-specific splits.

In [10]:
# Record the shared partition for every original row.
shared_split_assignment = pd.Series(
    index=table.index,
    dtype="string",
    name="partition",
)

shared_split_assignment.loc[training_indices] = "training"
shared_split_assignment.loc[validation_indices] = "validation"

assert shared_split_assignment.notna().all()

shared_split_manifest = pd.DataFrame({
    "id": table["id"],
    "partition": shared_split_assignment,
})
shared_split_manifest.index.name = "row_index"

# Use a separate directory for this experiment.
EXPERIMENT_DIR = PROJECT_ROOT / "results" / "06_shared_split"
EXPERIMENT_DIR.mkdir(parents=True, exist_ok=True)

split_path = EXPERIMENT_DIR / "shared_split.csv"

# Preserve any existing split and verify that it matches.
if split_path.exists():
    saved_split = pd.read_csv(
        split_path,
        index_col="row_index",
    )
    pd.testing.assert_frame_equal(
        saved_split,
        shared_split_manifest,
        check_dtype=False,
    )
    print("Existing shared split matches.")
else:
    shared_split_manifest.to_csv(split_path)
    print("Shared split saved.")

print("Path:", split_path)
display(shared_split_manifest["partition"].value_counts())

Existing shared split matches.
Path: C:\Users\EMS-User\capstone_virtual_lab_mohamed_suhmi\results\06_shared_split\shared_split.csv


partition
training      6378
validation    1595
Name: count, dtype: Int64

## 3. Prepare labeled training datasets for the individual models

For each target, select only training rows with an originally available
label. Prepare both the nine-descriptor and thirteen-descriptor feature sets.

Validation rows, identifiers, structure-group labels and other target
properties are excluded from the model inputs.
No missing target values are filled at this stage.

## 3. Erstellung gelabelter Trainingsdatensätze für die einzelnen Modelle

Wählen Sie für jedes Zielmerkmal (Target) ausschließlich diejenigen Trainingsdatensätze aus, für die ursprünglich ein Label vorliegt. Erstellen Sie jeweils einen Merkmalsdatensatz mit neun Deskriptoren sowie einen mit dreizehn Deskriptoren.

Validierungsdaten, Identifikatoren, Bezeichnungen von Strukturgruppen sowie weitere Eigenschaften der Zielmerkmale werden von den Modelleingabedaten ausgeschlossen.
Fehlende Zielwerte werden in dieser Phase nicht ergänzt.

In [11]:
# Make the project's reusable modules available.
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.polymer_modeling import FEATURE_SETS

TARGET_FULL_NAMES = {
    "Tg": "Glass transition temperature (Tg)",
    "FFV": "Fractional free volume (FFV)",
    "Tc": "Thermal conductivity (Tc)",
    "Density": "Mass density (Density)",
    "Rg": "Radius of gyration (Rg)",
}

stage1_training_data = {}
training_dataset_rows = []

for target in TARGETS:
    # Select observed labels within the shared training partition.
    observed_mask = y_training_original[target].notna()
    labeled_indices = y_training_original.index[observed_mask]

    y = y_training_original.loc[labeled_indices, target].copy()

    assert labeled_indices.intersection(validation_indices).empty
    assert np.isfinite(y.to_numpy()).all()

    stage1_training_data[target] = {}

    for feature_set_name, feature_columns in FEATURE_SETS.items():
        X = table.loc[labeled_indices, feature_columns].copy()

        assert X.index.equals(y.index)
        assert np.isfinite(X.to_numpy()).all()
        assert set(feature_columns).isdisjoint(TARGETS)

        stage1_training_data[target][feature_set_name] = {
            "X": X,
            "y": y.copy(),
        }

        training_dataset_rows.append({
            "target": TARGET_FULL_NAMES[target],
            "feature_set": feature_set_name,
            "training_labels": len(y),
            "features": X.shape[1],
        })

stage1_dataset_summary = pd.DataFrame(training_dataset_rows)

display(stage1_dataset_summary)

,target,feature_set,training_labels,features
0,Glass transition temperature (Tg),base9,424,9
1,Glass transition temperature (Tg),extended13,424,13
2,Fractional free volume (FFV),base9,5611,9
3,Fractional free volume (FFV),extended13,5611,13
4,Thermal conductivity (Tc),base9,592,9
5,Thermal conductivity (Tc),extended13,592,13
6,Mass density (Density),base9,490,9
7,Mass density (Density),extended13,490,13
8,Radius of gyration (Rg),base9,490,9
9,Radius of gyration (Rg),extended13,490,13


## 4. Cross-validation of the initial individual models

Compare four fixed model configurations with both feature sets.
Use five-fold cross-validation within the shared training partition.

For each target, all candidates use the same labeled rows and folds.
The primary metric is mean MAE. Mean fold-level R² is supplementary.
The outer validation partition remains unused.

In [12]:
from sklearn.model_selection import KFold, cross_validate
from src.polymer_modeling import make_model, MODEL_NAMES

N_SPLITS = 5

stage1_cv_folds = {}
stage1_cv_rows = []
stage1_cv_summary_rows = []

for target in TARGETS:
    # Create the folds once per target.
    reference = stage1_training_data[target]["base9"]

    folds = list(
        KFold(
            n_splits=N_SPLITS,
            shuffle=True,
            random_state=RANDOM_STATE,
        ).split(reference["X"])
    )

    stage1_cv_folds[target] = folds

    for feature_set_name, dataset in stage1_training_data[target].items():
        X = dataset["X"]
        y = dataset["y"]

        assert X.index.equals(reference["X"].index)
        pd.testing.assert_series_equal(y, reference["y"])

        for model_name in MODEL_NAMES:
            scores = cross_validate(
                estimator=make_model(model_name),
                X=X,
                y=y,
                cv=folds,
                scoring={
                    "mae": "neg_mean_absolute_error",
                    "r2": "r2",
                },
                n_jobs=1,
                error_score="raise",
            )

            fold_mae = -scores["test_mae"]
            fold_r2 = scores["test_r2"]

            for fold_number, (mae_value, r2_value) in enumerate(
                zip(fold_mae, fold_r2),
                start=1,
            ):
                stage1_cv_rows.append({
                    "target": target,
                    "feature_set": feature_set_name,
                    "model": model_name,
                    "fold": fold_number,
                    "mae": mae_value,
                    "r2": r2_value,
                })

            stage1_cv_summary_rows.append({
                "target": target,
                "feature_set": feature_set_name,
                "model": model_name,
                "mean_cv_mae": fold_mae.mean(),
                "std_cv_mae": fold_mae.std(ddof=1),
                "mean_fold_r2": fold_r2.mean(),
            })

        print(
            "Completed:",
            TARGET_FULL_NAMES[target],
            "/",
            feature_set_name,
        )

stage1_cv_fold_results = pd.DataFrame(stage1_cv_rows)

stage1_cv_summary = (
    pd.DataFrame(stage1_cv_summary_rows)
    .sort_values(["target", "mean_cv_mae"])
    .reset_index(drop=True)
)

# Save results separately from previous experiments.
stage1_cv_fold_results.to_csv(
    EXPERIMENT_DIR / "stage1_initial_cv_folds.csv",
    index=False,
)
stage1_cv_summary.to_csv(
    EXPERIMENT_DIR / "stage1_initial_cv_summary.csv",
    index=False,
)

# Display a compact comparison of mean MAE.
stage1_initial_mae_comparison = stage1_cv_summary.pivot(
    index=["target", "feature_set"],
    columns="model",
    values="mean_cv_mae",
).reindex(TARGETS, level="target")

display(stage1_initial_mae_comparison)

Completed: Glass transition temperature (Tg) / base9
Completed: Glass transition temperature (Tg) / extended13
Completed: Fractional free volume (FFV) / base9
Completed: Fractional free volume (FFV) / extended13
Completed: Thermal conductivity (Tc) / base9
Completed: Thermal conductivity (Tc) / extended13
Completed: Mass density (Density) / base9
Completed: Mass density (Density) / extended13
Completed: Radius of gyration (Rg) / base9
Completed: Radius of gyration (Rg) / extended13


model                   linear     median  random_forest    xgboost
target  feature_set                                                
Tg      base9        58.688328  87.823103      60.420171  62.052902
        extended13   58.911119  87.823103      60.532244  62.062758
FFV     base9         0.016700   0.020994       0.011294   0.012605
        extended13    0.016665   0.020994       0.010950   0.012083
Tc      base9         0.043124   0.076811       0.038523   0.038912
        extended13    0.043152   0.076811       0.038541   0.039447
Density base9         0.063923   0.108016       0.062244   0.056026
        extended13    0.064238   0.108016       0.062201   0.057460
Rg      base9         3.475446   3.814343       2.702539   2.639289
        extended13    3.479119   3.814343       2.705994   2.656000

### Findings: initial models on the shared training split

All learned models outperform the median baseline for every target.

The lowest mean cross-validation MAE among the fixed configurations
is achieved by:

- Glass transition temperature (Tg): linear regression, nine descriptors.
- Fractional free volume (FFV): Random Forest, thirteen descriptors.
- Thermal conductivity (Tc): Random Forest, nine descriptors.
- Mass density (Density): XGBoost, nine descriptors.
- Radius of gyration (Rg): XGBoost, nine descriptors.

For FFV, adding four descriptors reduces the Random Forest mean MAE
by approximately 3.05%. Effects on the other targets are small or mixed.

These rankings are preliminary and specific to the shared training
partition and evaluated configurations. The outer validation partition
has not been used in this comparison.

## 5. Reference results before descriptor expansion

Before testing additional molecular descriptors, we summarize the
initial cross-validation results from Section 4.

For each of the five target properties, we select the model and feature
set with the lowest mean cross-validation MAE among the configurations
tested so far.

The reference table includes:
- The selected model and feature set: 9 or 13 descriptors.
- The mean MAE across the five validation folds.
- The standard deviation of the fold MAE values.
- The mean R² across the five validation folds.

These results provide a reference for evaluating additional descriptors.
The upcoming comparisons will use the same training samples and
cross-validation folds.

**Evaluation scope:** These are development results used for model
selection. The shared outer validation subset is not used in this step.
No models are retrained when creating this summary.

In [13]:
# Select the best initial configuration for each target.
best_initial_indices = (
    stage1_cv_summary
    .groupby("target")["mean_cv_mae"]
    .idxmin()
)

stage1_reference = (
    stage1_cv_summary
    .loc[best_initial_indices]
    .set_index("target")
    .reindex(TARGETS)
    [
        [
            "feature_set",
            "model",
            "mean_cv_mae",
            "std_cv_mae",
            "mean_fold_r2",
        ]
    ]
)

# Save the reference results for the descriptor comparison.
stage1_reference.to_csv(
    EXPERIMENT_DIR / "stage1_reference_best_initial.csv"
)

display(stage1_reference.round(6))

,feature_set,model,mean_cv_mae,std_cv_mae,mean_fold_r2
target,,,,,
Tg,base9,linear,58.688328,4.530443,0.531246
FFV,extended13,random_forest,0.010950,0.000480,0.636853
Tc,base9,random_forest,0.038523,0.004631,0.630537
Density,base9,xgboost,0.056026,0.005679,0.514332
Rg,base9,xgboost,2.639289,0.225753,0.331476


## 6. Expanding the molecular descriptor set

We now test whether a broader description of polymer structure improves
the prediction of the five target properties.

We extend the existing descriptors with additional RDKit descriptors
covering molecular flexibility, ring structure, composition, surface
area, and structural complexity.

Before model training, we check the additional descriptors for missing
values, infinite values, and constant columns.

For a fair comparison, we retain the same training samples,
cross-validation folds, and initial model configurations.
The shared outer validation subset remains unused.

**Research question:** Do the additional structural descriptors reduce
cross-validation MAE compared with the existing 9- and 13-feature sets?

Molecular descriptors are calculated from the supplied SMILES
structures. They are not independently adjustable laboratory parameters.

## 6. Erweiterung des Satzes an Moleküldeskriptoren

Wir untersuchen nun, ob eine umfassendere Beschreibung der Polymerstruktur die Vorhersage der fünf Zielgrößen verbessert.

Wir ergänzen die bestehenden Deskriptoren um weitere RDKit-Deskriptoren, die Aspekte wie molekulare Flexibilität, Ringstruktur, Zusammensetzung, Oberfläche und strukturelle Komplexität abdecken.

Vor dem Training der Modelle prüfen wir die zusätzlichen Deskriptoren auf fehlende Werte, unendliche Werte sowie konstante Spalten.

Um einen fairen Vergleich zu gewährleisten, verwenden wir dieselben Trainingsstichproben, Kreuzvalidierungs-Folds und anfänglichen Modellkonfigurationen.
Die gemeinsam genutzte externe Validierungsmenge bleibt dabei unberücksichtigt.

**Forschungsfrage:** Führen die zusätzlichen strukturellen Deskriptoren zu einem geringeren MAE (Mean Absolute Error) in der Kreuzvalidierung im Vergleich zu den bereits vorhandenen Datensätzen mit 9 bzw. 13 Merkmalen?

Die Moleküldeskriptoren werden auf Basis der bereitgestellten SMILES-Strukturen berechnet. Es handelt sich dabei nicht um unabhängig einstellbare Laborparameter.

In [14]:
from rdkit import Chem
from rdkit.Chem import Descriptors

# Define eight additional descriptors before evaluating model performance.
ADDITIONAL_DESCRIPTORS = [
    "FractionCSP3",
    "NumRotatableBonds",
    "NumAliphaticRings",
    "NumSaturatedRings",
    "NumHeteroatoms",
    "LabuteASA",
    "BertzCT",
    "MolMR",
]

# Identify the original SMILES column.
smiles_columns = [
    column
    for column in table.columns
    if column.lower() == "smiles"
]

if len(smiles_columns) != 1:
    raise ValueError(
        f"Expected one SMILES column. Available columns: {table.columns.tolist()}"
    )

smiles_column = smiles_columns[0]

# Calculate descriptors without changing charges or attachment points.
additional_rows = []

for row_index, smiles in table[smiles_column].items():
    mol = Chem.MolFromSmiles(smiles) if isinstance(smiles, str) else None

    if mol is None:
        raise ValueError(f"Invalid or missing SMILES at row {row_index}")

    values = {
        name: float(getattr(Descriptors, name)(mol))
        for name in ADDITIONAL_DESCRIPTORS
    }
    additional_rows.append(values)

additional_descriptor_table = pd.DataFrame(
    additional_rows,
    index=table.index,
)

# Combine the existing 13 features with the eight new descriptors.
expanded21_features = pd.concat(
    [
        table[FEATURE_SETS["extended13"]].copy(),
        additional_descriptor_table,
    ],
    axis=1,
)

assert expanded21_features.index.equals(table.index)
assert expanded21_features.shape[1] == 21
assert expanded21_features.columns.is_unique
assert set(expanded21_features.columns).isdisjoint(TARGETS)

# Inspect descriptor quality using training rows only.
additional_training_features = additional_descriptor_table.loc[
    training_indices
]

additional_descriptor_checks = pd.DataFrame({
    "missing_count": additional_training_features.isna().sum(),
    "infinite_count": np.isinf(additional_training_features).sum(),
    "unique_count": additional_training_features.nunique(),
    "min": additional_training_features.min(),
    "max": additional_training_features.max(),
})

display(additional_descriptor_checks)

if not np.isfinite(
    additional_training_features.to_numpy(dtype=float)
).all():
    raise ValueError("Non-finite training descriptors require inspection.")

print("Expanded feature matrix:", expanded21_features.shape)
print(
    "Constant additional descriptors in training:",
    additional_descriptor_checks.index[
        additional_descriptor_checks["unique_count"] <= 1
    ].tolist(),
)

# Save the calculated features and training quality checks.
expanded21_features.to_csv(
    EXPERIMENT_DIR / "expanded21_features.csv",
    index_label="row_index",
)

additional_descriptor_checks.to_csv(
    EXPERIMENT_DIR / "additional_descriptor_checks.csv"
)

,missing_count,infinite_count,unique_count,min,max
FractionCSP3,0,0,510,0.000000,1.000000
NumRotatableBonds,0,0,52,0.000000,92.000000
NumAliphaticRings,0,0,11,0.000000,10.000000
NumSaturatedRings,0,0,10,0.000000,10.000000
NumHeteroatoms,0,0,38,2.000000,54.000000
LabuteASA,0,0,5296,8.739251,950.199587
BertzCT,0,0,5773,2.754888,8056.484946
MolMR,0,0,5543,4.807000,615.918000


Expanded feature matrix: (7973, 21)
Constant additional descriptors in training: []


## 7. Cross-validation with 21 molecular descriptors

We evaluate linear regression, Random Forest, and XGBoost using the
expanded set of 21 molecular descriptors.

For each target, we use exactly the same labelled training samples,
five cross-validation folds, and initial model configurations as in
Section 4. Only the input feature set changes.

We compare mean cross-validation MAE, its standard deviation, and
mean fold R² with the previous 9- and 13-feature results.

The median baseline is reused from Section 4 because its predictions
do not depend on the molecular descriptors.

The shared outer validation subset remains unused.

In [15]:
# Evaluate the expanded feature set using the existing CV folds.
expanded_cv_fold_rows = []
expanded_cv_summary_rows = []

for target in TARGETS:
    reference = stage1_training_data[target]["base9"]
    y = reference["y"]

    # Preserve exactly the same labelled rows and their order.
    X = expanded21_features.loc[reference["X"].index].copy()

    assert X.index.equals(y.index)
    assert X.index.intersection(validation_indices).empty
    assert np.isfinite(X.to_numpy(dtype=float)).all()

    folds = stage1_cv_folds[target]

    for model_name in ["linear", "random_forest", "xgboost"]:
        scores = cross_validate(
            estimator=make_model(model_name),
            X=X,
            y=y,
            cv=folds,
            scoring={
                "mae": "neg_mean_absolute_error",
                "r2": "r2",
            },
            n_jobs=1,
            error_score="raise",
        )

        fold_mae = -scores["test_mae"]
        fold_r2 = scores["test_r2"]

        for fold_number, (mae_value, r2_value) in enumerate(
            zip(fold_mae, fold_r2),
            start=1,
        ):
            expanded_cv_fold_rows.append({
                "target": target,
                "feature_set": "expanded21",
                "model": model_name,
                "fold": fold_number,
                "mae": mae_value,
                "r2": r2_value,
            })

        expanded_cv_summary_rows.append({
            "target": target,
            "feature_set": "expanded21",
            "model": model_name,
            "mean_cv_mae": fold_mae.mean(),
            "std_cv_mae": fold_mae.std(ddof=1),
            "mean_fold_r2": fold_r2.mean(),
        })

    print(f"Completed: {TARGET_FULL_NAMES[target]}")

expanded_cv_fold_results = pd.DataFrame(expanded_cv_fold_rows)
expanded_cv_summary = pd.DataFrame(expanded_cv_summary_rows)

# Combine previous and new results without changing the original tables.
descriptor_cv_summary = pd.concat(
    [stage1_cv_summary, expanded_cv_summary],
    ignore_index=True,
)

descriptor_mae_comparison = (
    descriptor_cv_summary.pivot(
        index=["target", "feature_set"],
        columns="model",
        values="mean_cv_mae",
    )
    .reindex(TARGETS, level="target")
    .reindex(columns=["linear", "random_forest", "xgboost"])
)

# Save the new results.
expanded_cv_fold_results.to_csv(
    EXPERIMENT_DIR / "stage1_expanded21_cv_folds.csv",
    index=False,
)

descriptor_cv_summary.to_csv(
    EXPERIMENT_DIR / "stage1_descriptor_cv_summary.csv",
    index=False,
)

display(descriptor_mae_comparison.round(6))

Completed: Glass transition temperature (Tg)
Completed: Fractional free volume (FFV)
Completed: Thermal conductivity (Tc)
Completed: Mass density (Density)
Completed: Radius of gyration (Rg)


model                   linear  random_forest    xgboost
target  feature_set                                     
Tg      base9        58.688328      60.420171  62.052902
        expanded21   59.475265      56.986610  57.473914
        extended13   58.911119      60.532244  62.062758
FFV     base9         0.016700       0.011294   0.012605
        expanded21    0.014629       0.009608   0.009937
        extended13    0.016665       0.010950   0.012083
Tc      base9         0.043124       0.038523   0.038912
        expanded21    0.036132       0.033463   0.032419
        extended13    0.043152       0.038541   0.039447
Density base9         0.063923       0.062244   0.056026
        expanded21    0.052600       0.043009   0.040973
        extended13    0.064238       0.062201   0.057460
Rg      base9         3.475446       2.702539   2.639289
        expanded21    3.133757       2.586909   2.555244
        extended13    3.479119       2.705994   2.656000

### Findings from the descriptor expansion

Expanding the descriptor set to 21 features reduced the best mean
cross-validation MAE for all five target properties compared with
the best previous configuration using 9 or 13 features.

The approximate reductions were:
- Glass transition temperature (Tg): 2.91%.
- Fractional free volume (FFV): 12.26%.
- Thermal conductivity (Tc): 15.85%.
- Mass density (Density): 26.87%.
- Radius of gyration (Rg): 3.19%.

Random Forest achieved the lowest mean CV MAE for Tg and FFV.
XGBoost achieved the lowest mean CV MAE for Tc, Density, and Rg.

The strongest improvement was observed for mass density.
Additional descriptors did not improve every individual model;
their benefit depended on the target and model family.

These comparisons used identical training samples, CV folds, and
fixed model configurations. Percentages are approximate because
they were calculated from the displayed rounded values.

These are development CV results used for configuration selection.
They do not establish statistical significance or independently
validated performance. The shared outer validation subset was
not used in this comparison.

In [16]:
# Select the best model using 21 descriptors for each target.
best_expanded_indices = (
    expanded_cv_summary
    .groupby("target")["mean_cv_mae"]
    .idxmin()
)

best_expanded = (
    expanded_cv_summary
    .loc[best_expanded_indices]
    .set_index("target")
    .reindex(TARGETS)
)

model_labels = {
    "median": "Median baseline",
    "linear": "Linear regression",
    "random_forest": "Random Forest",
    "xgboost": "XGBoost",
}

feature_counts = {
    "base9": 9,
    "extended13": 13,
    "expanded21": 21,
}

# Align previous and expanded results by target.
descriptor_results = pd.DataFrame(index=stage1_reference.index)

descriptor_results["Target property"] = [
    TARGET_FULL_NAMES[target]
    for target in descriptor_results.index
]

descriptor_results["Previous best model"] = (
    stage1_reference["model"].map(model_labels)
)

descriptor_results["Previous features"] = (
    stage1_reference["feature_set"].map(feature_counts)
)

descriptor_results["Previous CV MAE"] = stage1_reference["mean_cv_mae"]

descriptor_results["Best model: 21 features"] = (
    best_expanded["model"].map(model_labels)
)

descriptor_results["CV MAE: 21 features"] = best_expanded["mean_cv_mae"]

descriptor_results["MAE reduction (%)"] = (
    (
        descriptor_results["Previous CV MAE"]
        - descriptor_results["CV MAE: 21 features"]
    )
    / descriptor_results["Previous CV MAE"]
    * 100
)

# Save full numerical precision; round only for display.
descriptor_results.to_csv(
    EXPERIMENT_DIR / "descriptor_expansion_results.csv",
    index_label="target",
)

display(
    descriptor_results
    .set_index("Target property")
    .style
    .format({
        "Previous features": "{:.0f}",
        "Previous CV MAE": "{:.6f}",
        "CV MAE: 21 features": "{:.6f}",
        "MAE reduction (%)": "{:.2f}%",
    })
    .set_caption(
        "Descriptor expansion: best previous configuration vs. "
        "best configuration with 21 features — five-fold development CV"
    )
)

,Previous best model,Previous features,Previous CV MAE,Best model: 21 features,CV MAE: 21 features,MAE reduction (%)
Target property,,,,,,
Glass transition temperature (Tg),Linear regression,9,58.688328,Random Forest,56.986610,2.90%
Fractional free volume (FFV),Random Forest,13,0.010950,Random Forest,0.009608,12.25%
Thermal conductivity (Tc),Random Forest,9,0.038523,XGBoost,0.032419,15.84%
Mass density (Density),XGBoost,9,0.056026,XGBoost,0.040973,26.87%
Radius of gyration (Rg),XGBoost,9,2.639289,XGBoost,2.555244,3.18%


## 8. Paired comparison across cross-validation folds

We compare the previous best configuration with the best configuration
using 21 descriptors on the same five validation folds.

For each target, we report the MAE of both configurations in each fold
and count how often the expanded configuration achieves a lower MAE.

This describes the consistency of the observed improvement.
It is not a statistical significance test. For some targets, both
the feature set and the selected model family change.

In [17]:
paired_fold_tables = []
fold_consistency_rows = []

for target in TARGETS:
    previous = stage1_reference.loc[target]
    expanded = best_expanded.loc[target]

    # Select the previous best configuration's fold results.
    previous_folds = (
        stage1_cv_fold_results.loc[
            (stage1_cv_fold_results["target"] == target)
            & (
                stage1_cv_fold_results["feature_set"]
                == previous["feature_set"]
            )
            & (stage1_cv_fold_results["model"] == previous["model"]),
            ["fold", "mae"],
        ]
        .rename(columns={"mae": "previous_mae"})
    )

    # Select the best expanded configuration's fold results.
    expanded_folds = (
        expanded_cv_fold_results.loc[
            (expanded_cv_fold_results["target"] == target)
            & (expanded_cv_fold_results["model"] == expanded["model"]),
            ["fold", "mae"],
        ]
        .rename(columns={"mae": "expanded21_mae"})
    )

    # Match results explicitly by fold number.
    paired = previous_folds.merge(
        expanded_folds,
        on="fold",
        how="outer",
        validate="one_to_one",
    ).sort_values("fold")

    assert len(paired) == N_SPLITS
    assert not paired.isna().any().any()

    paired["mae_difference"] = (
        paired["expanded21_mae"] - paired["previous_mae"]
    )

    paired.insert(0, "target", target)
    paired_fold_tables.append(paired)

    fold_consistency_rows.append({
        "Target property": TARGET_FULL_NAMES[target],
        "Improved folds": int((paired["mae_difference"] < 0).sum()),
        "Worse folds": int((paired["mae_difference"] > 0).sum()),
        "Equal folds": int((paired["mae_difference"] == 0).sum()),
        "Total folds": len(paired),
    })

paired_descriptor_folds = pd.concat(
    paired_fold_tables,
    ignore_index=True,
)

descriptor_fold_consistency = pd.DataFrame(fold_consistency_rows)

paired_descriptor_folds.to_csv(
    EXPERIMENT_DIR / "descriptor_paired_folds.csv",
    index=False,
)

descriptor_fold_consistency.to_csv(
    EXPERIMENT_DIR / "descriptor_fold_consistency.csv",
    index=False,
)

display(descriptor_fold_consistency)

,Target property,Improved folds,Worse folds,Equal folds,Total folds
0,Glass transition temperature (Tg),3,2,0,5
1,Fractional free volume (FFV),5,0,0,5
2,Thermal conductivity (Tc),5,0,0,5
3,Mass density (Density),5,0,0,5
4,Radius of gyration (Rg),4,1,0,5


### Interpretation of the paired fold comparison

The selected 21-descriptor configurations achieved lower MAE in:
- 3 of 5 folds for glass transition temperature (Tg).
- 5 of 5 folds for fractional free volume (FFV).
- 5 of 5 folds for thermal conductivity (Tc).
- 5 of 5 folds for mass density (Density).
- 4 of 5 folds for radius of gyration (Rg).

For FFV, Tc, and Density, the improvement was observed in every fold.
For Tg and Rg, the average improvements were smaller and were not
observed in every fold.

These are descriptive comparisons of configurations selected using
the same development CV results. The folds have overlapping training
sets, and the comparison does not establish statistical significance.
For Tg and Tc, the selected model family also changes.

## 9. Morgan fingerprints for Random Forest

We test Morgan fingerprints as an alternative to the molecular
descriptor sets.

Fingerprint settings:
- 1,024 binary positions.
- Radius 2.
- Chirality information included where specified in the input SMILES.
- Original charges and attachment points retained.

In this experiment, fingerprints are used on their own, without
appending the 21 molecular descriptors.

Random Forest will be evaluated using the same labelled training
samples, CV folds, and initial model configuration as before.
Hyperparameter optimization follows the representation comparison.

Fingerprint generation uses molecular structures only.
The shared outer validation subset remains unused for model selection.

In [18]:
from rdkit.Chem import rdFingerprintGenerator

# Fix fingerprint settings before evaluating model performance.
MORGAN_BITS = 1024
MORGAN_RADIUS = 2
MORGAN_INCLUDE_CHIRALITY = True

morgan_generator = rdFingerprintGenerator.GetMorganGenerator(
    radius=MORGAN_RADIUS,
    fpSize=MORGAN_BITS,
    includeChirality=MORGAN_INCLUDE_CHIRALITY,
    countSimulation=False,
)

# Convert each original SMILES into a binary fingerprint.
morgan_rows = []

for row_index, smiles in table[smiles_column].items():
    mol = Chem.MolFromSmiles(smiles) if isinstance(smiles, str) else None

    if mol is None:
        raise ValueError(f"Invalid or missing SMILES at row {row_index}")

    fingerprint = morgan_generator.GetFingerprintAsNumPy(mol)
    morgan_rows.append(fingerprint)

# Preserve the original row identifiers and use compact integer storage.
morgan1024_features = pd.DataFrame(
    np.stack(morgan_rows).astype(np.uint8),
    index=table.index,
    columns=[
        f"Morgan_{position:04d}"
        for position in range(MORGAN_BITS)
    ],
)

assert morgan1024_features.index.equals(table.index)
assert morgan1024_features.shape == (len(table), MORGAN_BITS)
assert morgan1024_features.isin([0, 1]).all().all()

# Inspect fingerprints on training rows only.
morgan_training = morgan1024_features.loc[training_indices]
active_bits_per_polymer = morgan_training.sum(axis=1)

print("Fingerprint matrix:", morgan1024_features.shape)
print("Missing training values:", int(morgan_training.isna().sum().sum()))
print(
    "Constant positions in training:",
    int((morgan_training.nunique() <= 1).sum()),
)

display(
    active_bits_per_polymer
    .describe()
    .to_frame(name="Active fingerprint positions per training polymer")
)

# Save fingerprints together with their generation settings.
morgan1024_features.to_pickle(
    EXPERIMENT_DIR / "morgan1024_features.pkl"
)

pd.Series({
    "radius": MORGAN_RADIUS,
    "fp_size": MORGAN_BITS,
    "include_chirality": MORGAN_INCLUDE_CHIRALITY,
    "count_simulation": False,
}).to_json(
    EXPERIMENT_DIR / "morgan1024_settings.json",
    indent=2,
)

Fingerprint matrix: (7973, 1024)
Missing training values: 0
Constant positions in training: 1


,Active fingerprint positions per training polymer
count,6378.000000
mean,37.574788
std,12.810098
min,4.000000
25%,28.000000
50%,36.000000
75%,46.000000
max,115.000000


## 10. Random Forest: descriptors versus Morgan fingerprints

We evaluate Random Forest using the 1,024-position Morgan fingerprints.

For each target, we retain the same labelled training rows, row order,
five CV folds, and initial Random Forest configuration used previously.

We compare the fingerprint results with Random Forest using
9, 13, and 21 molecular descriptors. Only the input representation changes.

Mean CV MAE is the primary comparison metric; lower values are better.
The shared outer validation subset remains unused.

In [19]:
# Evaluate Random Forest using Morgan fingerprints.
morgan_cv_fold_rows = []
morgan_cv_summary_rows = []

for target in TARGETS:
    reference = stage1_training_data[target]["base9"]
    y = reference["y"]

    X = morgan1024_features.loc[reference["X"].index].copy()

    assert X.index.equals(y.index)
    assert X.index.intersection(validation_indices).empty

    scores = cross_validate(
        estimator=make_model("random_forest"),
        X=X,
        y=y,
        cv=stage1_cv_folds[target],
        scoring={
            "mae": "neg_mean_absolute_error",
            "r2": "r2",
        },
        n_jobs=1,
        error_score="raise",
    )

    fold_mae = -scores["test_mae"]
    fold_r2 = scores["test_r2"]

    for fold_number, (mae_value, r2_value) in enumerate(
        zip(fold_mae, fold_r2),
        start=1,
    ):
        morgan_cv_fold_rows.append({
            "target": target,
            "feature_set": "morgan1024",
            "model": "random_forest",
            "fold": fold_number,
            "mae": mae_value,
            "r2": r2_value,
        })

    morgan_cv_summary_rows.append({
        "target": target,
        "feature_set": "morgan1024",
        "model": "random_forest",
        "mean_cv_mae": fold_mae.mean(),
        "std_cv_mae": fold_mae.std(ddof=1),
        "mean_fold_r2": fold_r2.mean(),
    })

    print(f"Completed: {TARGET_FULL_NAMES[target]}")

morgan_cv_fold_results = pd.DataFrame(morgan_cv_fold_rows)
morgan_cv_summary = pd.DataFrame(morgan_cv_summary_rows)

# Combine all representation results.
representation_cv_summary = pd.concat(
    [descriptor_cv_summary, morgan_cv_summary],
    ignore_index=True,
)

# Compare the same model across the four representations.
rf_representation_comparison = (
    representation_cv_summary.loc[
        representation_cv_summary["model"] == "random_forest"
    ]
    .pivot(
        index="target",
        columns="feature_set",
        values="mean_cv_mae",
    )
    .reindex(
        index=TARGETS,
        columns=["base9", "extended13", "expanded21", "morgan1024"],
    )
)

morgan_cv_fold_results.to_csv(
    EXPERIMENT_DIR / "stage1_morgan1024_rf_cv_folds.csv",
    index=False,
)

representation_cv_summary.to_csv(
    EXPERIMENT_DIR / "stage1_representation_cv_summary.csv",
    index=False,
)

display(rf_representation_comparison.round(6))

Completed: Glass transition temperature (Tg)
Completed: Fractional free volume (FFV)
Completed: Thermal conductivity (Tc)
Completed: Mass density (Density)
Completed: Radius of gyration (Rg)


feature_set,base9,extended13,expanded21,morgan1024
target,,,,
Tg,60.420171,60.532244,56.986610,56.037183
FFV,0.011294,0.010950,0.009608,0.009173
Tc,0.038523,0.038541,0.033463,0.033318
Density,0.062244,0.062201,0.043009,0.052165
Rg,2.702539,2.705994,2.586909,1.767211


### Findings: descriptors versus Morgan fingerprints

With the fixed Random Forest configuration, Morgan fingerprints
achieved lower mean CV MAE than the 21-descriptor set for Tg, FFV,
Tc, and Rg. For Density, the 21-descriptor set performed better.

The largest fingerprint benefit was observed for Rg:
approximately 31.69% lower MAE than Random Forest with 21 descriptors.

Across all configurations tested so far:
- Random Forest with Morgan fingerprints performed best for Tg, FFV,
  and Rg.
- XGBoost with 21 descriptors performed best for Tc and Density.

The preferred molecular representation therefore depends on the target.
These are development cross-validation results used for selection;
independent confirmation remains necessary.

## 11. Targeted hyperparameter optimization for Rg

We tune Random Forest with Morgan fingerprints for radius of gyration
(Rg), using a small randomized search.

Evaluation procedure:
- Five existing outer CV folds.
- Six candidate configurations per outer fold.
- Three inner CV folds for hyperparameter selection.
- Selection criterion: mean absolute error.

The selected configuration is refitted on each outer training fold
and evaluated on its held-out fold.

The number of trees remains fixed at 200 to limit computation.
We vary feature subsampling, tree depth, and minimum leaf size.

The molecular representation was selected using earlier development
CV results. This nested evaluation separates hyperparameter tuning
from scoring, but does not independently validate that earlier selection.

The shared outer validation subset remains unused.

In [20]:
from sklearn.model_selection import RandomizedSearchCV

# Use the same labelled Rg rows and their original order.
rg_reference = stage1_training_data["Rg"]["base9"]
X_rg_morgan = morgan1024_features.loc[rg_reference["X"].index].copy()
y_rg = rg_reference["y"]

assert X_rg_morgan.index.equals(y_rg.index)
assert X_rg_morgan.index.intersection(validation_indices).empty

# Define a small search space.
rg_parameter_space = {
    "max_features": [0.1, 0.3, 0.6, 1.0],
    "max_depth": [None, 12, 24],
    "min_samples_leaf": [1, 2, 4],
}

rg_inner_cv = KFold(
    n_splits=3,
    shuffle=True,
    random_state=RANDOM_STATE,
)

rg_search = RandomizedSearchCV(
    estimator=make_model("random_forest").set_params(n_estimators=200),
    param_distributions=rg_parameter_space,
    n_iter=6,
    scoring="neg_mean_absolute_error",
    cv=rg_inner_cv,
    refit=True,
    random_state=RANDOM_STATE,
    n_jobs=1,
    error_score="raise",
)

# Evaluate the entire search procedure on the existing outer folds.
rg_nested_scores = cross_validate(
    estimator=rg_search,
    X=X_rg_morgan,
    y=y_rg,
    cv=stage1_cv_folds["Rg"],
    scoring={
        "mae": "neg_mean_absolute_error",
        "r2": "r2",
    },
    return_estimator=True,
    n_jobs=1,
    verbose=2,
    error_score="raise",
)

rg_tuned_fold_mae = -rg_nested_scores["test_mae"]

# Match the initial Random Forest results by fold number.
rg_initial_folds = (
    morgan_cv_fold_results.loc[
        morgan_cv_fold_results["target"] == "Rg"
    ]
    .set_index("fold")
    .reindex(range(1, N_SPLITS + 1))
)

assert not rg_initial_folds["mae"].isna().any()

rg_tuning_comparison = pd.DataFrame({
    "fold": range(1, N_SPLITS + 1),
    "initial_mae": rg_initial_folds["mae"].to_numpy(),
    "tuned_mae": rg_tuned_fold_mae,
})

rg_tuning_comparison["mae_difference"] = (
    rg_tuning_comparison["tuned_mae"]
    - rg_tuning_comparison["initial_mae"]
)

rg_tuning_summary = pd.DataFrame({
    "configuration": ["Initial RF + Morgan", "Tuned RF + Morgan"],
    "mean_cv_mae": [
        rg_initial_folds["mae"].mean(),
        rg_tuned_fold_mae.mean(),
    ],
    "std_cv_mae": [
        rg_initial_folds["mae"].std(ddof=1),
        rg_tuned_fold_mae.std(ddof=1),
    ],
})

# Record the configuration selected within each outer training fold.
rg_selected_parameters = pd.DataFrame([
    {"fold": fold, **search.best_params_}
    for fold, search in enumerate(
        rg_nested_scores["estimator"],
        start=1,
    )
])

rg_tuning_comparison.to_csv(
    EXPERIMENT_DIR / "rg_morgan_tuning_folds.csv",
    index=False,
)

rg_tuning_summary.to_csv(
    EXPERIMENT_DIR / "rg_morgan_tuning_summary.csv",
    index=False,
)

rg_selected_parameters.to_csv(
    EXPERIMENT_DIR / "rg_morgan_parameters_by_fold.csv",
    index=False,
)

display(rg_tuning_comparison.round(6))
display(rg_tuning_summary.round(6))
display(rg_selected_parameters)

[CV] END .................................................... total time=  14.0s
[CV] END .................................................... total time=  13.7s
[CV] END .................................................... total time=  13.4s
[CV] END .................................................... total time=  13.4s
[CV] END .................................................... total time=  13.7s


[Parallel(n_jobs=1)]: Done   5 out of   5 | elapsed:  1.1min finished


,fold,initial_mae,tuned_mae,mae_difference
0,1,1.919224,1.891664,-0.027560
1,2,1.467400,1.438443,-0.028957
2,3,1.885907,1.941951,0.056044
3,4,1.611018,1.547744,-0.063275
4,5,1.952506,1.959072,0.006566


,configuration,mean_cv_mae,std_cv_mae
0,Initial RF + Morgan,1.767211,0.215531
1,Tuned RF + Morgan,1.755775,0.244148


,fold,min_samples_leaf,max_features,max_depth
0,1,1,0.6,24
1,2,1,0.6,24
2,3,2,0.6,24
3,4,2,0.6,24
4,5,1,0.6,24


### Findings from targeted Rg tuning

Nested hyperparameter tuning reduced mean CV MAE from 1.767211
to 1.755775, an approximate reduction of 0.65%.

The tuned procedure improved MAE in three of five outer folds.
The standard deviation of fold MAE increased from 0.215531 to 0.244148.
The average benefit was therefore small and not consistent across folds.

All five inner searches selected max_features=0.6 and max_depth=24.
The selected min_samples_leaf values varied between 1 and 2.

For this experiment, changing the molecular representation produced
a much larger improvement than the subsequent small hyperparameter search.

These remain development results. The nested procedure evaluates
hyperparameter selection within the previously selected representation;
the shared outer validation subset remains unused.